In [ ]:
%matplotlib widget

# KuroSiwo Dataset: Exploratory Data Analysis

KuroSiwo is a large-scale, multi-temporal SAR dataset for flood detection with 43 real flood events, 1.73M catalogue entries, and ~1.6M exported patches.

## Key terminology

| Term | Definition |
|------|-----------|
| **Event** (`actid`) | A single flood case. The dataset contains 43 events (2015–2022). |
| **Tile** (`grid_id`) | A unique spatial location on the ground (224x224 px). Each tile belongs to exactly one event. An event comprises 22–43,349 tiles. |
| **Patch** (= 1 row) | A tile observed at a specific time and product type. One tile produces 1–3 patches: pre-flood GRD, pre-flood SLC, and/or flood-time GRD. A patch is what gets exported to disk as a `.tif`. |
| **`pflood`** | Ground-truth flood extent label (0–100%) per tile. Static across time — identical for pre-flood and flood-time patches of the same tile. Used as ML training target. |
| **`master`** | Temporal role flag. `True` = flood-time acquisition, `False` = pre-flood baseline. |
| **`crank`** | Product type. `1` = GRD (amplitude), `2` = SLC (complex). |

## Notebook structure

1. **Dataset overview** — shape, product types, temporal roles, quality flags
2. **Data model** — how tiles, patches, and events relate; spatial extent analysis
3. **Flood labels (`pflood`)** — distribution, class imbalance, critical finding (static label)
4. **Temporal structure** — acquisition dates, pre→flood ordering, gaps
5. **Metadata extraction** — per-event bounding boxes, dates, flood extent for partner delivery


In [ ]:
%reset -f
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import subprocess
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
# ============================================================================
# SETUP: Data Download & Configuration
# ============================================================================
# NOTE: Original catalogue URL (Dropbox):
# "https://www.dropbox.com/scl/fi/wu6nvj73cz4h7k3gxpzx6/catalogue.gpkg"
# ?rlkey=hsij2o0k60r2n0z6z4d2ngww9&st=0zjqhzgx&dl=1

from pathlib import Path
import shutil

catalogue_path = Path("../../assets/ks_catalogue.gpkg")

def is_lfs_pointer(path: Path) -> bool:
    try:
        with open(path, 'r', encoding='utf8') as fh:
            first = fh.readline()
        return first.startswith('version https://git-lfs.github.com/spec')
    except Exception:
        return False

def run_cmd(cmd, cwd=None):
    try:
        subprocess.run(cmd, check=True, cwd=cwd)
        return True
    except FileNotFoundError:
        return False
    except subprocess.CalledProcessError:
        return False

# Ensure the asset is present and not an LFS pointer; try to fetch if needed
if catalogue_path.exists():
    if is_lfs_pointer(catalogue_path):
        try:
            subprocess.run(['git', 'lfs', 'pull'], check=True)
        except subprocess.CalledProcessError:
            raise RuntimeError(
                "`git lfs pull` failed; probably git lfs is not installed. "
                "Try: git lfs install, then retry."
            )
        else:
            print(f"Catalogue ready at {catalogue_path}")
else:
    raise Exception(
        f"Catalogue not found at {catalogue_path}, and no way to fetch it (git lfs not available)"
    )

gdf = gpd.read_file(str(catalogue_path))

In [ ]:
print(f"Loaded: {gdf.shape[0]:,} rows × {gdf.shape[1]} columns")
print(f"\nAll columns:\n{gdf.columns.tolist()}")
gdf.crs

In [ ]:
pd.set_option('display.max_colwidth', None)
gdf.sample(3)

## 1. Dataset Overview

Top-level shape: how many rows, events, product types, and quality coverage?


In [ ]:
# How many events, patches and what time span does the catalogue cover?
print(f"Total catalogue rows:          {len(gdf):,}")
print(f"Unique spatial patches:        {gdf['grid_id'].nunique():,}")
print(f"Unique flood events (actid):   {gdf['actid'].nunique()}")
print(f"Date range:                    {gdf['flood_date'].min().date()} → {gdf['flood_date'].max().date()}")
print(f"Source Date range:             {gdf['source_date'].min().date()} → {gdf['source_date'].max().date()}")

mask_source_after_flood = pd.to_datetime(gdf['source_date']) > pd.to_datetime(gdf['flood_date'])
n_source_after_flood = mask_source_after_flood.sum()
print(f"Rows with source_date > flood_date: {n_source_after_flood:,} ({100*n_source_after_flood/len(gdf):.2f}%)")

In [ ]:
# What product types are available? crank=1 → GRD (amplitude), crank=2 → SLC (complex)
for crank, label in {1: 'GRD (amplitude)', 2: 'SLC (complex)'}.items():
    n = (gdf['crank'] == crank).sum()
    print(f"  crank={crank}  {label:20s}  {n:>10,}  ({100*n/len(gdf):.1f}%)")

In [ ]:
# What is the temporal role of each patch? master=True → flood-time, master=False → pre-flood baseline
n_flood    = (gdf['master'] == True).sum()
n_preflood = (gdf['master'] == False).sum()
print(f"  master=True  (flood-time):       {n_flood:>10,}  ({100*n_flood/len(gdf):.1f}%)")
print(f"  master=False (pre-flood baseline):{n_preflood:>10,}  ({100*n_preflood/len(gdf):.1f}%)")

In [ ]:
# How many patches are actually exported to disk and pass quality validation?
n_exported = (gdf['exported'] == True).sum()
n_valid    = (gdf['gvalid']   == True).sum()
print(f"  Exported (on disk):       {n_exported:>10,}  ({100*n_exported/len(gdf):.1f}%)")
print(f"  Quality valid (gvalid):   {n_valid:>10,}  ({100*n_valid/len(gdf):.1f}%)")

## 2. Flood Labels (`pflood`)

Each patch carries a `pflood` value: the percentage of the 256×256 pixel tile covered by flood water. This is a **static spatial label** — identical for pre-flood and flood-time patches of the same tile (see Section 4 for verification).


In [ ]:
# Flood label statistics — flood-time patches only (master=True, GRD product)
flood_patches = gdf[(gdf['master'] == True) & (gdf['crank'] == 1)]
s = flood_patches['pflood']
print(f"Flood-time patches (GRD):  {len(flood_patches):,}")
print(f"pflood range:              {s.min():.1f}% – {s.max():.1f}%")
print(f"Mean / Median:             {s.mean():.2f}% / {s.median():.2f}%")

In [ ]:
# Class breakdown — among patches that HAVE a flood label (pflood is not NaN)
s_labeled = s.dropna()
n_labeled = len(s_labeled)
n_unlabeled = len(s) - n_labeled
print(f"Flood-time patches with a pflood label:    {n_labeled:>8,}  ({100*n_labeled/len(s):.1f}%)")
print(f"Flood-time patches without label (NaN):    {n_unlabeled:>8,}  ({100*n_unlabeled/len(s):.1f}%)")
print()
no_flood  = (s_labeled == 0).sum()
low       = ((s_labeled > 0)  & (s_labeled <= 5)).sum()
mid       = ((s_labeled > 5)  & (s_labeled <= 50)).sum()
high      = (s_labeled > 50).sum()
for label, n in [('No flood (0%)', no_flood), ('Low (0–5%)', low), ('Significant (5–50%)', mid), ('Dominant (>50%)', high)]:
    print(f"  {label:25s}  {n:>8,}  ({100*n/n_labeled:.1f}% of labeled)")

In [ ]:
# Visualise the pflood distribution and class imbalance (labeled patches only)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(s_labeled, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(s_labeled.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {s_labeled.median():.1f}%')
ax1.set_xlabel('Flood coverage per patch (%)')
ax1.set_ylabel('Number of patches')
ax1.set_title('Distribution of pflood (labeled patches)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

counts = [no_flood, low, mid, high]
labels = ['No flood\n(0%)', 'Low\n(0–5%)', 'Significant\n(5–50%)', 'Dominant\n(>50%)']
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#c0392b']
ax2.pie(counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax2.set_title('Class distribution (labeled flood-time patches)')

plt.tight_layout()
plt.show()

## 3. Data Model: Tiles, Patches, Events

How does the catalogue map from rows (patches) to spatial tiles (`grid_id`) to flood events (`actid`)?

**Hierarchy:** EVENT (actid) → many TILES (grid_id) → each tile has 1–3 PATCHES (rows in the catalogue, one per temporal role × product type)


In [ ]:
# Build a per-event summary (GRD patches only, crank=1)
records = []
for actid in sorted(gdf['actid'].unique()):
    ev = gdf[gdf['actid'] == actid]
    grd = ev[ev['crank'] == 1]
    records.append({
        'event_id': actid,
        'flood_date': grd[grd['master'] == True]['flood_date'].iloc[0].date(),
        'flood_patches': (grd['master'] == True).sum(),
        'preflood_patches': (grd['master'] == False).sum(),
        'valid_exported': grd[(grd['exported'] == True) & (grd['gvalid'] == True)].shape[0],
        'labeled_flood': grd[(grd['master'] == True) & (grd['pflood'] > 0)].shape[0],
    })
event_meta_df = pd.DataFrame(records)
print(f"Events with pre-flood baseline: {(event_meta_df['preflood_patches'] > 0).sum()} / {len(event_meta_df)}")
print(f"Total flood patches:            {event_meta_df['flood_patches'].sum():,}")
print(f"Total pre-flood patches:        {event_meta_df['preflood_patches'].sum():,}")
print(f"Total valid+exported patches:   {event_meta_df['valid_exported'].sum():,}")
print(f"Patches with flood label >0%:   {event_meta_df['labeled_flood'].sum():,}")

In [ ]:
# Which events are the largest? Top 15 by valid exported patch count
top15 = event_meta_df.nlargest(15, 'valid_exported')
print("Top 15 events by valid exported patches:")
print(top15[['event_id', 'flood_date', 'flood_patches', 'preflood_patches', 'valid_exported', 'labeled_flood']].to_string(index=False))

In [ ]:
# Are labeled patches proportional to event size? (sanity check)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

top15_plot = event_meta_df.nlargest(15, 'valid_exported')
ax1.barh(range(len(top15_plot)), top15_plot['valid_exported'], color='steelblue', alpha=0.7)
ax1.set_yticks(range(len(top15_plot)))
ax1.set_yticklabels([f"Event {e}" for e in top15_plot['event_id']], fontsize=9)
ax1.set_xlabel('Valid exported patches')
ax1.set_title('Top 15 events by patch count')
ax1.grid(axis='x', alpha=0.3)

ax2.scatter(event_meta_df['valid_exported'], event_meta_df['labeled_flood'],
            s=60, alpha=0.7, color='coral', edgecolors='black', linewidth=0.4)
max_v = event_meta_df['valid_exported'].max()
ax2.plot([0, max_v], [0, max_v], 'k--', alpha=0.3, linewidth=1, label='1:1 line')
ax2.set_xlabel('Valid exported patches')
ax2.set_ylabel('Patches with flood label (pflood > 0%)')
ax2.set_title('Labeled coverage per event')
ax2.set_ylim(0, 8000)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# PATCHES → EVENTS: in-depth consistency checks
# ═══════════════════════════════════════════════════════════════════════════════
# Questions:
#   1. Do all patches within an (event, master, source_date) group share the same date?
#   2. Are pre-flood dates ALWAYS before flood-time dates?
#   3. How does pflood vary across patches within an event?
#   4. How many unique acquisition dates per event × role?

gdf['source_date'] = pd.to_datetime(gdf['source_date'])
gdf['flood_date'] = pd.to_datetime(gdf['flood_date'])

# ── Q1: Date consistency within acquisitions ──────────────────────────────────
# For each (actid, master, crank), how many distinct source_dates exist?
date_counts = (
    gdf.groupby(['actid', 'master', 'crank'])['source_date']
    .nunique()
    .reset_index(name='n_dates')
)
print("═══ Q1: Date consistency per (event, role, product) ═══")
print(f"  Groups with exactly 1 source_date: {(date_counts['n_dates'] == 1).sum()}")
print(f"  Groups with 2 source_dates:        {(date_counts['n_dates'] == 2).sum()}")
print(f"  Groups with 3+ source_dates:       {(date_counts['n_dates'] >= 3).sum()}")
print(f"\n  → Multiple dates per group means multiple acquisitions for the same role.")
print(f"     This is expected: KuroSiwo uses ~2 pre-flood baselines per event.")
print()

# Detail: per-event breakdown of acquisition dates
acq_structure = (
    gdf[gdf['crank'] == 1]
    .groupby(['actid', 'master'])['source_date']
    .agg(['nunique', 'min', 'max'])
    .reset_index()
)
acq_structure.columns = ['actid', 'master', 'n_acq_dates', 'earliest', 'latest']
print("Per-event acquisition structure (GRD only):")
print(f"  Pre-flood (master=False):")
pre_struct = acq_structure[acq_structure['master'] == False]
print(f"    Acquisition dates per event: min={pre_struct['n_acq_dates'].min()}, "
      f"max={pre_struct['n_acq_dates'].max()}, "
      f"median={pre_struct['n_acq_dates'].median():.0f}")
flood_struct = acq_structure[acq_structure['master'] == True]
print(f"  Flood-time (master=True):")
print(f"    Acquisition dates per event: min={flood_struct['n_acq_dates'].min()}, "
      f"max={flood_struct['n_acq_dates'].max()}, "
      f"median={flood_struct['n_acq_dates'].median():.0f}")

# ── Q2: Are pre-flood dates ALWAYS before flood-time dates? ───────────────────
print("\n═══ Q2: Temporal ordering — pre-flood always before flood-time? ═══")
violations = []
for eid in sorted(gdf['actid'].unique()):
    ev = gdf[(gdf['actid'] == eid) & (gdf['crank'] == 1)]
    pre_max = ev[ev['master'] == False]['source_date'].max()
    flood_min = ev[ev['master'] == True]['source_date'].min()
    if pd.isna(pre_max) or pd.isna(flood_min):
        continue
    if pre_max >= flood_min:
        violations.append({'actid': eid, 'pre_max': pre_max.date(),
                           'flood_min': flood_min.date(),
                           'overlap_days': (pre_max - flood_min).days})

if violations:
    print(f"  ⚠ {len(violations)} events have pre-flood dates >= flood-time dates!")
    print(pd.DataFrame(violations).to_string(index=False))
else:
    print("  ✓ All 43 events: last pre-flood date is strictly BEFORE first flood-time date.")

# Also check: is source_date always consistent with the master label semantics?
# i.e., do any pre-flood patches have source_date AFTER the event's flood_date?
pre_after_flood = gdf[(gdf['master'] == False) & (gdf['source_date'] > gdf['flood_date'])]
flood_before_flood = gdf[(gdf['master'] == True) & (gdf['source_date'] < gdf['flood_date'])]
print(f"\n  Pre-flood patches with source_date > flood_date: {len(pre_after_flood):,} "
      f"({100*len(pre_after_flood)/len(gdf[gdf['master']==False]):.1f}% of pre-flood)")
print(f"  Flood-time patches with source_date < flood_date: {len(flood_before_flood):,} "
      f"({100*len(flood_before_flood)/len(gdf[gdf['master']==True]):.1f}% of flood-time)")
if len(pre_after_flood) > 0 or len(flood_before_flood) > 0:
    print("  → NOTE: source_date vs flood_date ordering is NOT a strict rule.")
    print("    master=True/False is the authoritative role label, not date ordering vs flood_date.")


In [ ]:

# ── Q3: pflood variation WITHIN an event ──────────────────────────────────────
# pflood is a static label (same for pre/flood), so we look at its spatial
# distribution across patches within each event.

print("═══ Q3: pflood variability across patches within each event ═══\n")

# Per-event pflood stats (GRD, all patches regardless of master)
event_pflood = (
    gdf[gdf['crank'] == 1]
    .groupby('actid')['pflood']
    .agg(['count', 'mean', 'std', 'min', 'median', 'max',
          lambda x: (x > 0).sum(), lambda x: (x > 0).mean() * 100])
    .reset_index()
)
event_pflood.columns = ['actid', 'n_patches', 'mean', 'std', 'min', 'median',
                         'max', 'n_nonzero', 'pct_nonzero']

print("Per-event pflood statistics (GRD patches, both pre & flood combined):")
print(f"  Mean pflood across events:   {event_pflood['mean'].mean():.2f}% "
      f"(range: {event_pflood['mean'].min():.2f}% – {event_pflood['mean'].max():.2f}%)")
print(f"  Std of pflood within events: {event_pflood['std'].mean():.2f}% "
      f"(range: {event_pflood['std'].min():.2f}% – {event_pflood['std'].max():.2f}%)")
print(f"  % patches with pflood > 0:   {event_pflood['pct_nonzero'].mean():.1f}% "
      f"(range: {event_pflood['pct_nonzero'].min():.1f}% – {event_pflood['pct_nonzero'].max():.1f}%)")

# Show the distribution
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))

# Left: mean pflood per event
ax1.bar(range(len(event_pflood)), event_pflood.sort_values('mean', ascending=False)['mean'],
        color='steelblue', edgecolor='black', linewidth=0.3, alpha=0.8)
ax1.set_xlabel('Events (sorted)')
ax1.set_ylabel('Mean pflood (%)')
ax1.set_title('Mean pflood per event')
ax1.set_xticks([])
ax1.grid(axis='y', alpha=0.3)

# Middle: % of patches with pflood > 0
ax2.bar(range(len(event_pflood)),
        event_pflood.sort_values('pct_nonzero', ascending=False)['pct_nonzero'],
        color='coral', edgecolor='black', linewidth=0.3, alpha=0.8)
ax2.set_xlabel('Events (sorted)')
ax2.set_ylabel('% patches with pflood > 0')
ax2.set_title('Fraction of patches labelled as flooded')
ax2.set_xticks([])
ax2.grid(axis='y', alpha=0.3)

# Right: pflood std (within-event variability)
ax3.bar(range(len(event_pflood)),
        event_pflood.sort_values('std', ascending=False)['std'],
        color='mediumpurple', edgecolor='black', linewidth=0.3, alpha=0.8)
ax3.set_xlabel('Events (sorted)')
ax3.set_ylabel('pflood std (%)')
ax3.set_title('Within-event pflood variability (std)')
ax3.set_xticks([])
ax3.grid(axis='y', alpha=0.3)

plt.suptitle('pflood varies spatially across patches but is STATIC across time',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Detailed table: top 10 events by mean pflood
print("\nTop 10 events by mean pflood:")
top10 = event_pflood.nlargest(10, 'mean')[
    ['actid', 'n_patches', 'mean', 'std', 'pct_nonzero', 'max']
].copy()
top10.columns = ['actid', 'n_patches', 'mean_%', 'std_%', 'pct_nonzero_%', 'max_%']
print(top10.to_string(index=False, float_format='%.1f'))


In [ ]:

# ── pflood vs pwater: permanent water and flood labelling ─────────────────────
# pwater = % of tile covered by permanent water bodies (rivers, lakes)
# pflood = % of tile covered by flood water
# Both are static spatial labels per tile. How do they relate?

print("═══ pwater vs pflood: permanent water & flood labels ═══\n")

# Ensure gdf_area exists (equal-area projection for km² calculations)
AREA_CRS = 'EPSG:6933'
gdf_area = gdf.to_crs(AREA_CRS).copy()
gdf_area['patch_area_km2'] = gdf_area.geometry.area / 1_000_000

# Work with one row per tile (GRD, flood-time) to avoid double-counting
tiles = gdf[(gdf['crank'] == 1) & (gdf['master'] == True)].drop_duplicates(subset='grid_id').copy()
both_available = tiles[tiles['pwater'].notna() & tiles['pflood'].notna()]

print(f"Tiles with both pwater & pflood labels: {len(both_available):,} / {len(tiles):,}")
print(f"  pwater coverage: {tiles['pwater'].notna().sum():,} tiles ({100*tiles['pwater'].notna().mean():.1f}%)")
print(f"  pflood coverage: {tiles['pflood'].notna().sum():,} tiles ({100*tiles['pflood'].notna().mean():.1f}%)")
print()

# Categorise tiles by water type
pw = both_available['pwater']
pf = both_available['pflood']

no_water     = ((pw == 0) & (pf == 0)).sum()
flood_only   = ((pw == 0) & (pf > 0)).sum()
perm_only    = ((pw > 0) & (pf == 0)).sum()
both_nonzero = ((pw > 0) & (pf > 0)).sum()

print("Tile categories (labeled tiles only):")
print(f"  Dry (no water, no flood):        {no_water:>8,}  ({100*no_water/len(both_available):.1f}%)")
print(f"  Flood only (no permanent water): {flood_only:>8,}  ({100*flood_only/len(both_available):.1f}%)")
print(f"  Permanent water only (no flood): {perm_only:>8,}  ({100*perm_only/len(both_available):.1f}%)")
print(f"  Both flood + permanent water:    {both_nonzero:>8,}  ({100*both_nonzero/len(both_available):.1f}%)")
print(f"\nCorrelation (pflood, pwater): {pf.corr(pw):.3f}")

# ── Assign spatial category to each tile for mapping ──────────────────────────
both_geo = both_available.to_crs('EPSG:4326').copy()
conditions = [
    (both_geo['pwater'] == 0) & (both_geo['pflood'] == 0),
    (both_geo['pwater'] == 0) & (both_geo['pflood'] > 0),
    (both_geo['pwater'] > 0)  & (both_geo['pflood'] == 0),
    (both_geo['pwater'] > 0)  & (both_geo['pflood'] > 0),
]
labels_cat = ['Dry land', 'Flood only', 'Permanent water only', 'Flood + permanent water']
both_geo['category'] = np.select(conditions, labels_cat, default='Unknown')

# ── Visualisation ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 12))

# (a) Geographic map: tile centroids coloured by water category
ax1 = fig.add_subplot(2, 2, 1)
cat_colors = {
    'Dry land': '#d4c5a9',
    'Flood only': '#e74c3c',
    'Permanent water only': '#2980b9',
    'Flood + permanent water': '#8e44ad',
}
# Plot centroids (use representative_point to avoid geographic CRS warning)
centroids = both_geo.copy()
rep_points = centroids.geometry.representative_point()
centroids['cx'] = rep_points.x
centroids['cy'] = rep_points.y
# Show each category in order (dry first so it's background)
for cat in ['Dry land', 'Permanent water only', 'Flood only', 'Flood + permanent water']:
    subset = centroids[centroids['category'] == cat]
    ax1.scatter(subset['cx'], subset['cy'], s=1.5, alpha=0.4,
                color=cat_colors[cat], label=f'{cat} ({len(subset):,})', rasterized=True)
ax1.set_xlabel('Longitude (°)')
ax1.set_ylabel('Latitude (°)')
ax1.set_title('(a) Geographic distribution of tile water types')
ax1.legend(fontsize=7, markerscale=5, loc='lower left', framealpha=0.9)
ax1.grid(alpha=0.2)

# (b) Per-event: flood extent vs permanent water extent (km²)
ax2 = fig.add_subplot(2, 2, 2)
event_water = []
for eid in sorted(both_available['actid'].unique()):
    ev = gdf_area[(gdf_area['actid'] == eid) & (gdf_area['crank'] == 1) & (gdf_area['master'] == True)]
    ev = ev.drop_duplicates(subset='grid_id')
    ev_labeled = ev[ev['pwater'].notna() & ev['pflood'].notna()]
    if len(ev_labeled) == 0:
        continue
    flood_area = (ev_labeled['patch_area_km2'] * ev_labeled['pflood'] / 100).sum()
    water_area = (ev_labeled['patch_area_km2'] * ev_labeled['pwater'] / 100).sum()
    event_water.append({'actid': eid, 'flood_km2': flood_area, 'perm_water_km2': water_area})
ew_df = pd.DataFrame(event_water)

ax2.scatter(ew_df['perm_water_km2'], ew_df['flood_km2'],
            s=70, color='steelblue', edgecolors='black', linewidths=0.5, alpha=0.8)
for _, row in ew_df.iterrows():
    if row['flood_km2'] > 500 or row['perm_water_km2'] > 100:
        ax2.annotate(f"{int(row['actid'])}", (row['perm_water_km2'], row['flood_km2']),
                     fontsize=7, ha='left', va='bottom', color='dimgray')
ax2.set_xlabel('Permanent water extent (km²)')
ax2.set_ylabel('Flood extent (km²)')
ax2.set_title('(b) Flood vs permanent water per event')
ax2.grid(alpha=0.3)

# (c) 2D density: pflood vs pwater (hexbin — intuitive heatmap)
ax3 = fig.add_subplot(2, 2, 3)
hb = ax3.hexbin(pw, pf, gridsize=30, cmap='YlOrRd', mincnt=1,
                bins='log', linewidths=0.2, edgecolors='gray')
cb = plt.colorbar(hb, ax=ax3, label='log₁₀(tile count)')
ax3.set_xlabel('pwater — permanent water (%)')
ax3.set_ylabel('pflood — flood extent (%)')
ax3.set_title('(c) Joint density: where do tiles cluster?')
ax3.set_xlim(-2, 102)
ax3.set_ylim(-2, 102)

# (d) Per-event bar: % of flood extent over permanent water vs dry land
ax4 = fig.add_subplot(2, 2, 4)
event_overlap = []
for eid in sorted(both_available['actid'].unique()):
    ev = gdf_area[(gdf_area['actid'] == eid) & (gdf_area['crank'] == 1) & (gdf_area['master'] == True)]
    ev = ev.drop_duplicates(subset='grid_id')
    ev_labeled = ev[ev['pwater'].notna() & ev['pflood'].notna() & (ev['pflood'] > 0)]
    if len(ev_labeled) == 0:
        continue
    flood_over_water = (ev_labeled[ev_labeled['pwater'] > 0]['patch_area_km2']
                        * ev_labeled[ev_labeled['pwater'] > 0]['pflood'] / 100).sum()
    flood_over_dry = (ev_labeled[ev_labeled['pwater'] == 0]['patch_area_km2']
                      * ev_labeled[ev_labeled['pwater'] == 0]['pflood'] / 100).sum()
    total = flood_over_water + flood_over_dry
    if total > 0:
        event_overlap.append({
            'actid': eid,
            'pct_over_water': 100 * flood_over_water / total,
            'pct_over_dry': 100 * flood_over_dry / total,
        })
eo_df = pd.DataFrame(event_overlap).sort_values('pct_over_water', ascending=False).reset_index(drop=True)

ax4.barh(range(len(eo_df)), eo_df['pct_over_water'], color='#2980b9',
         alpha=0.8, label='Over permanent water')
ax4.barh(range(len(eo_df)), eo_df['pct_over_dry'],
         left=eo_df['pct_over_water'], color='#e74c3c',
         alpha=0.8, label='Over dry land')
ax4.set_xlabel('% of flood extent')
ax4.set_ylabel('Events (sorted)')
ax4.set_yticks([])
ax4.set_xlim(0, 100)
ax4.set_title('(d) Flood extent: over water bodies vs dry land')
ax4.legend(fontsize=8, loc='lower right')
ax4.grid(axis='x', alpha=0.3)

plt.suptitle('Permanent water (pwater) vs flood extent (pflood): spatial context',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary
print("\n═══ Key insight ═══")
print(f"  {100*both_nonzero/(flood_only+both_nonzero):.0f}% of flooded tiles also have permanent water")
print(f"  → Most flooding occurs near existing water bodies (riverine/fluvial floods)")
print(f"  {100*flood_only/(flood_only+both_nonzero):.0f}% of flooded tiles are over normally dry land")
print(f"  → Significant pluvial/flash flood component in the dataset")
print(f"\n  Median % of event flood extent over permanent water: "
      f"{eo_df['pct_over_water'].median():.0f}%")


In [ ]:

# ── Q4: Patch-to-event mapping — are patches uniquely assigned? ───────────────
print("═══ Q4: Patch → Event mapping ═══\n")

# Does the same grid_id ever appear in multiple events?
grid_events = gdf.groupby('grid_id')['actid'].nunique()
shared_grids = grid_events[grid_events > 1]
print(f"  Unique grid_ids in catalogue: {len(grid_events):,}")
print(f"  grid_ids appearing in >1 event: {len(shared_grids):,} "
      f"({100*len(shared_grids)/len(grid_events):.2f}%)")
if len(shared_grids) > 0:
    print(f"  Max events per grid_id: {shared_grids.max()}")
    print(f"  → Some tiles are reused across events (overlapping AOIs)")

# Within a single event: does a grid_id appear multiple times?
# Expected: once per (master, crank) combination = up to 4 rows per grid_id per event
rows_per_grid_event = gdf.groupby(['actid', 'grid_id']).size()
print(f"\n  Rows per (event, grid_id): min={rows_per_grid_event.min()}, "
      f"max={rows_per_grid_event.max()}, median={rows_per_grid_event.median():.0f}")
print(f"  Distribution:")
for n, cnt in rows_per_grid_event.value_counts().sort_index().items():
    print(f"    {n} rows: {cnt:,} grid_ids ({100*cnt/len(rows_per_grid_event):.1f}%)")

# Do all patches within a (event, master, crank, source_date) have the SAME pflood?
# This tests whether pflood is truly per-tile or if it varies by something else
pflood_per_tile = (
    gdf[gdf['crank'] == 1]
    .groupby(['actid', 'grid_id'])['pflood']
    .agg(['nunique', 'std'])
    .reset_index()
)
pflood_varies = pflood_per_tile[pflood_per_tile['nunique'] > 1]
print(f"\n  Grid_ids with varying pflood across roles within same event: "
      f"{len(pflood_varies):,} / {len(pflood_per_tile):,}")
if len(pflood_varies) == 0:
    print("  ✓ pflood is perfectly consistent per (event, grid_id) — truly a spatial label.")
else:
    print(f"  ⚠ {len(pflood_varies)} tiles have different pflood for pre vs flood!")

# ── Summary: how patches aggregate to events ──────────────────────────────────
print("\n═══ SUMMARY: Patches → Events ═══")
print("""
Each event (actid) consists of:
  • A set of spatial tiles (grid_id), each 256×256 px
  • Each tile appears in multiple temporal roles:
    - master=False: pre-flood baseline (typically 2 acquisition dates)
    - master=True:  flood-time (typically 1 acquisition date)
  • Each tile × role can have GRD (crank=1) and/or SLC (crank=2) products
  • pflood is a SPATIAL label per tile — same value regardless of temporal role
  • To aggregate patches → event-level metadata:
    - Bounding box: union of all tile geometries
    - Flood extent: sum(pflood × tile_area) over flood-time GRD tiles with pflood > 0
    - Date range: min(source_date where master=False) → min(source_date where master=True)
""")


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# SPATIAL EXTENT ANALYSIS: tile sizes and event footprints
# ═══════════════════════════════════════════════════════════════════════════════
# Are all tiles the same size? (Hint: NO — SAR pixel size varies with latitude)

# Reproject to equal-area CRS for accurate area calculations
AREA_CRS = 'EPSG:6933'
gdf_area = gdf.to_crs(AREA_CRS)

# Tile (grid_id) sizes in equal-area CRS
gdf_area['tile_area_km2'] = gdf_area.geometry.area / 1_000_000

# Since each grid_id appears 1-3 times, take one row per grid_id
tile_areas = gdf_area.drop_duplicates(subset='grid_id')['tile_area_km2']

print("═══ Tile (grid_id) spatial sizes ═══")
print(f"  Number of unique tiles: {len(tile_areas):,}")
print(f"  Area range: {tile_areas.min():.4f} – {tile_areas.max():.4f} km²")
print(f"  Mean: {tile_areas.mean():.4f} km²  |  Median: {tile_areas.median():.4f} km²")
print(f"  Std:  {tile_areas.std():.4f} km²  |  CV: {100*tile_areas.std()/tile_areas.mean():.1f}%")
print(f"\n  → Tiles are NOT uniform in size (CV={100*tile_areas.std()/tile_areas.mean():.1f}%).")
print(f"    This is expected: SAR ground-range pixel size varies with latitude and incidence angle.")
print(f"    A 256×256 pixel tile maps to different physical areas depending on location.")

# Per-event footprint (bounding box in equal-area CRS)
event_extents = []
for eid in sorted(gdf_area['actid'].unique()):
    ev = gdf_area[gdf_area['actid'] == eid]
    xmin, ymin, xmax, ymax = ev.total_bounds
    n_tiles = ev['grid_id'].nunique()
    total_tile_area = ev.drop_duplicates(subset='grid_id')['tile_area_km2'].sum()
    bbox_area = (xmax - xmin) * (ymax - ymin) / 1e6
    event_extents.append({
        'actid': eid, 'n_tiles': n_tiles,
        'total_tile_area_km2': total_tile_area,
        'bbox_area_km2': bbox_area,
        'width_km': (xmax - xmin) / 1000,
        'height_km': (ymax - ymin) / 1000,
        'fill_ratio': total_tile_area / bbox_area if bbox_area > 0 else 0,
    })
extent_df = pd.DataFrame(event_extents)

print(f"\n═══ Per-event spatial footprint (EPSG:6933) ═══")
print(f"  Tiles per event:       min={extent_df['n_tiles'].min():,}, "
      f"max={extent_df['n_tiles'].max():,}, median={extent_df['n_tiles'].median():.0f}")
print(f"  Total tile area (km²): min={extent_df['total_tile_area_km2'].min():.0f}, "
      f"max={extent_df['total_tile_area_km2'].max():.0f}, "
      f"median={extent_df['total_tile_area_km2'].median():.0f}")
print(f"  Bbox area (km²):       min={extent_df['bbox_area_km2'].min():.0f}, "
      f"max={extent_df['bbox_area_km2'].max():.0f}, "
      f"median={extent_df['bbox_area_km2'].median():.0f}")
print(f"  Fill ratio (tiles/bbox): min={extent_df['fill_ratio'].min():.3f}, "
      f"max={extent_df['fill_ratio'].max():.3f}, "
      f"median={extent_df['fill_ratio'].median():.3f}")
print(f"\n  → Fill ratio < 1 means tiles don't fully cover the bbox (gaps/irregular tiling).")

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) Tile area distribution
ax = axes[0, 0]
ax.hist(tile_areas, bins=50, color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(tile_areas.median(), color='red', linestyle='--', linewidth=1.5,
           label=f'Median: {tile_areas.median():.4f} km²')
ax.set_xlabel('Tile area (km²)')
ax.set_ylabel('Number of tiles')
ax.set_title('(a) Individual tile sizes vary with latitude')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# (b) Event footprint area vs number of tiles
ax = axes[0, 1]
ax.scatter(extent_df['n_tiles'], extent_df['total_tile_area_km2'],
           s=50, color='coral', edgecolors='black', linewidths=0.4, alpha=0.8)
ax.set_xlabel('Number of tiles (grid_ids) per event')
ax.set_ylabel('Total tile area (km²)')
ax.set_title('(b) Event size: more tiles = larger coverage')
ax.grid(alpha=0.3)

# (c) Event dimensions (width × height)
ax = axes[1, 0]
ax.scatter(extent_df['width_km'], extent_df['height_km'],
           s=50, color='mediumpurple', edgecolors='black', linewidths=0.4, alpha=0.8)
ax.plot([0, 700], [0, 700], 'k--', alpha=0.3, linewidth=1)
ax.set_xlabel('Event width (km)')
ax.set_ylabel('Event height (km)')
ax.set_title('(c) Event footprint dimensions')
ax.set_xlim(0)
ax.set_ylim(0)
ax.grid(alpha=0.3)

# (d) Fill ratio distribution
ax = axes[1, 1]
ax.hist(extent_df['fill_ratio'], bins=20, color='seagreen', edgecolor='black', alpha=0.8)
ax.set_xlabel('Fill ratio (sum of tile areas / bbox area)')
ax.set_ylabel('Number of events')
ax.set_title('(d) How densely tiles fill the event bbox')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Spatial extent analysis: tiles are variable-sized, events span 12–648 km',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table of extremes
print("\nLargest 5 events by tile count:")
print(extent_df.nlargest(5, 'n_tiles')[
    ['actid', 'n_tiles', 'total_tile_area_km2', 'width_km', 'height_km', 'fill_ratio']
].to_string(index=False, float_format='%.1f'))
print("\nSmallest 5 events by tile count:")
print(extent_df.nsmallest(5, 'n_tiles')[
    ['actid', 'n_tiles', 'total_tile_area_km2', 'width_km', 'height_km', 'fill_ratio']
].to_string(index=False, float_format='%.1f'))


## 5. Metadata Extraction

To run model equivalents we need per-event: `flood_case`, `date_start`, `date_end`, and `bounding_box(lat_min, lat_max, lon_min, lon_max)`. Useful extras: `max_flood_extent_km²` and `date_of_max_flood_extent`.


In [ ]:

# Reproject once for geographic bounds and once for area-accurate extent calculations
AREA_CRS = 'EPSG:6933'  # Equal-area projection for km² area estimates
EXTENT_CRANK = 1        # Use GRD patches for extent consistency
EXTENT_PRODUCT = {1: 'GRD', 2: 'SLC'}[EXTENT_CRANK]

gdf_wgs84 = gdf.to_crs('EPSG:4326')
gdf_area = gdf.to_crs(AREA_CRS).copy()
gdf_area['patch_area_km2'] = gdf_area.geometry.area / 1_000_000

case_records = []
for actid in sorted(gdf['actid'].unique()):
    ev_orig = gdf[gdf['actid'] == actid]
    ev_geo = gdf_wgs84[gdf_wgs84['actid'] == actid]
    ev_area = gdf_area[gdf_area['actid'] == actid]

    flood_rows = ev_orig[ev_orig['master'] == True]
    preflood_rows = ev_orig[ev_orig['master'] == False]

    lon_min, lat_min, lon_max, lat_max = ev_geo.total_bounds

    flooded = ev_area[
        (ev_area['master'] == True)
        & (ev_area['crank'] == EXTENT_CRANK)
        & (ev_area['pflood'].notna())
        & (ev_area['pflood'] > 0)
    ]
    extent_km2 = (flooded['patch_area_km2'] * flooded['pflood'] / 100).sum()

    # date_start = earliest pre-flood SAR acquisition (source_date for master=False)
    # date_end   = flood-time SAR acquisition (source_date for master=True)
    date_start = pd.to_datetime(preflood_rows['source_date']).min().date() if len(preflood_rows) else None
    date_end   = pd.to_datetime(flood_rows['source_date']).min().date()    if len(flood_rows)    else None

    case_records.append({
        'flood_case': f"KuroSiwo_{actid:03d}",
        'date_start': date_start,
        'date_end': date_end,
        'lat_min': round(lat_min, 4), 'lat_max': round(lat_max, 4),
        'lon_min': round(lon_min, 4), 'lon_max': round(lon_max, 4),
        'max_flood_extent_km2': round(extent_km2, 1),
        'date_of_max_flood_extent': date_end,
    })

flood_case_df = pd.DataFrame(case_records)
print(f"Extracted metadata for {len(flood_case_df)} flood cases.")
print(f"Extent method: sum(pflood_fraction × per-patch area in {AREA_CRS}, {EXTENT_PRODUCT} only)")


In [ ]:
# Sample the metadata table — do all required fields look correct?
print(flood_case_df[['flood_case', 'date_start', 'date_end', 'lat_min', 'lat_max', 'lon_min', 'lon_max']].to_string(index=False))

In [ ]:
# When did flood events occur? (temporal distribution by year)
years = pd.to_datetime(flood_case_df['date_end']).dt.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(years.index, years.values, color='steelblue', alpha=0.8, edgecolor='black')
ax.set_xlabel('Year')
ax.set_ylabel('Number of flood cases')
ax.set_title('Flood cases by year')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# How large are the bounding boxes? (degrees)
lat_span = flood_case_df['lat_max'] - flood_case_df['lat_min']
lon_span = flood_case_df['lon_max'] - flood_case_df['lon_min']
print(f"Latitude span  — mean: {lat_span.mean():.2f}°, median: {lat_span.median():.2f}°, range: {lat_span.min():.2f}° – {lat_span.max():.2f}°")
print(f"Longitude span — mean: {lon_span.mean():.2f}°, median: {lon_span.median():.2f}°, range: {lon_span.min():.2f}° – {lon_span.max():.2f}°")

In [ ]:
# How large is the flooded area per event? (computed from pflood × per-patch equal-area geometry)
ext = flood_case_df['max_flood_extent_km2']
print(f"Max flood extent per case — mean: {ext.mean():.0f} km², median: {ext.median():.0f} km², range: {ext.min():.1f} – {ext.max():.0f} km²")

In [ ]:
# Where are the flood cases located and how big are their footprints?
vis = flood_case_df.copy()
vis['lat_span'] = vis['lat_max'] - vis['lat_min']
vis['lon_span'] = vis['lon_max'] - vis['lon_min']
vis['center_lon'] = (vis['lon_min'] + vis['lon_max']) / 2
vis['center_lat'] = (vis['lat_min'] + vis['lat_max']) / 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

sc = ax1.scatter(vis['center_lon'], vis['center_lat'],
                 s=80, c=vis['max_flood_extent_km2'], cmap='YlOrRd',
                 edgecolors='black', linewidth=0.4, alpha=0.8)
plt.colorbar(sc, ax=ax1, label='Max flood extent (km²)')
ax1.set_xlabel('Longitude (°)')
ax1.set_ylabel('Latitude (°)')
ax1.set_title('Geographic distribution of flood cases')
ax1.grid(alpha=0.3)

CUTOFF = 1500
ext_all = vis['max_flood_extent_km2']
in_range = ext_all[ext_all <= CUTOFF]
outliers = ext_all[ext_all > CUTOFF]

ax2.hist(in_range, bins=15, color='coral', edgecolor='black', alpha=0.7)
ax2.set_xlim(0, CUTOFF)
ax2.set_xlabel('Max flood extent (km²)')
ax2.set_ylabel('Number of cases')
ax2.set_title('Distribution of max flood extent')
ax2.grid(axis='y', alpha=0.3)

if len(outliers):
    ax2.annotate(
        '···',
        xy=(1.0, 0), xycoords='axes fraction',
        xytext=(4, -18), textcoords='offset points',
        fontsize=13, color='dimgray', annotation_clip=False,
    )
    ax2.annotate(
        f'+ {len(outliers)} cases (>{CUTOFF:,} km², max {int(outliers.max()):,} km²)',
        xy=(0.98, 0.97), xycoords='axes fraction',
        ha='right', va='top', fontsize=8, color='dimgray', style='italic',
    )

plt.tight_layout()
plt.show()

In [ ]:
# # Export the dataframe to csv
flood_case_df.to_csv("kurosiwo_metadata_v1.csv", index=False)

In [ ]:

# Sanity check: date_start should be BEFORE date_end for all cases
cols = ['flood_case', 'date_start', 'date_end', 'date_of_max_flood_extent']
print(flood_case_df[cols].head(10).to_string(index=False))
equal = (flood_case_df['date_start'] == flood_case_df['date_end']).sum()
print(f"\nCases where date_start == date_end: {equal}")


## 4. Temporal Structure & Critical Findings

How does KuroSiwo structure each flood event in time? Key questions:
- How many acquisition dates per event and role?
- Are pre-flood dates always before flood-time?
- How large are the temporal gaps?
- Is `pflood` a temporal signal or a static label?


In [ ]:

# ── Panel 1: conceptual timeline schematic for a single typical event ──────────
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

fig, axes = plt.subplots(1, 2, figsize=(14, 4),
                         gridspec_kw={'width_ratios': [1, 2]})

# ── LEFT: schematic ────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(-1, 2)
ax.axis('off')
ax.set_title('Typical event structure (schematic)', fontsize=11, fontweight='bold')

# timeline backbone
ax.annotate('', xy=(9.5, 0.5), xytext=(0.5, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.text(9.7, 0.5, 'time', va='center', fontsize=9)

# known acquisition markers
known_x = [1.5, 3.0, 8.0]
known_labels = ['Pre-flood\nacq. 1\n(master=False)', 'Pre-flood\nacq. 2\n(master=False)',
                'Flood-time\nacq.\n(master=True)']
known_colors = ['#2980b9', '#2980b9', '#e74c3c']
for x, lbl, col in zip(known_x, known_labels, known_colors):
    ax.plot(x, 0.5, 'o', color=col, markersize=10, zorder=3)
    ax.text(x, -0.1, lbl, ha='center', va='top', fontsize=7.5, color=col)

# unknown flood window shading
ax.axvspan(3.0, 8.0, ymin=0.3, ymax=0.7, alpha=0.12, color='orange')
ax.annotate('', xy=(7.5, 1.2), xytext=(3.5, 1.2),
            arrowprops=dict(arrowstyle='<->', color='darkorange', lw=1.2))
ax.text(5.5, 1.35, 'Flood duration\n(unknown)', ha='center', fontsize=8,
        color='darkorange', style='italic')

# unknown peak marker
ax.plot(5.5, 0.5, 'x', color='darkorange', markersize=11, markeredgewidth=2, zorder=3)
ax.text(5.5, 0.75, '? peak', ha='center', fontsize=8, color='darkorange', style='italic')

legend_handles = [
    mlines.Line2D([], [], color='#2980b9', marker='o', linestyle='None',
                  markersize=8, label='Known SAR acquisition (pre-flood)'),
    mlines.Line2D([], [], color='#e74c3c', marker='o', linestyle='None',
                  markersize=8, label='Known SAR acquisition (flood-time)'),
    mlines.Line2D([], [], color='darkorange', marker='x', linestyle='None',
                  markersize=8, markeredgewidth=2, label='Unknown (onset / peak / end)'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=7.5, framealpha=0.8)

# ── RIGHT: all-events Gantt — pre-flood window + flood-time marker ─────────────
ax2 = axes[1]
ax2.set_title('All 43 events: pre-flood window & flood-time acquisition', fontsize=11,
              fontweight='bold')

# build per-event timeline from gdf
timeline = []
for eid in sorted(gdf['actid'].unique()):
    ev = gdf[gdf['actid'] == eid]
    pre_dates = pd.to_datetime(ev[ev['master'] == False]['source_date']).dropna()
    post_dates = pd.to_datetime(ev[ev['master'] == True]['source_date']).dropna()
    if pre_dates.empty or post_dates.empty:
        continue
    timeline.append({
        'event': f"{eid}",
        'pre_start': pre_dates.min(),
        'pre_end': pre_dates.max(),
        'flood_date': post_dates.min(),
    })
tdf = pd.DataFrame(timeline).sort_values('flood_date').reset_index(drop=True)

for i, row in tdf.iterrows():
    # pre-flood window bar
    ax2.barh(i, (row['pre_end'] - row['pre_start']).days,
             left=row['pre_start'], height=0.5,
             color='#2980b9', alpha=0.6)
    # gap between last pre-flood and flood-time (unknown flood window)
    gap_days = (row['flood_date'] - row['pre_end']).days
    ax2.barh(i, gap_days, left=row['pre_end'], height=0.5,
             color='orange', alpha=0.25)
    # flood-time acquisition
    ax2.plot(row['flood_date'], i, 'o', color='#e74c3c', markersize=5, zorder=3)

ax2.set_yticks(range(len(tdf)))
ax2.set_yticklabels([f"Event {e}" for e in tdf['event']], fontsize=7)
ax2.set_xlabel('Date')
ax2.grid(axis='x', alpha=0.3)

legend2 = [
    mpatches.Patch(color='#2980b9', alpha=0.6, label='Pre-flood acquisitions window'),
    mpatches.Patch(color='orange', alpha=0.4, label='Gap to flood image (flood duration unknown)'),
    mlines.Line2D([], [], color='#e74c3c', marker='o', linestyle='None',
                  markersize=6, label='Flood-time acquisition (= date_end)'),
]
ax2.legend(handles=legend2, loc='lower right', fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.show()


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# CRITICAL FINDING: pflood is a STATIC SPATIAL LABEL, not a temporal signal
# ═══════════════════════════════════════════════════════════════════════════════
# pflood is IDENTICAL for pre-flood and flood-time acquisitions within each event.
# It represents the ground-truth flood mask fraction per tile, applied uniformly
# to all temporal acquisitions (pre and post) as a label for ML training.

# Verify: compare mean pflood of pre-flood vs flood-time per event (GRD only)
grd = gdf[gdf['crank'] == 1].copy()
results = []
for eid in sorted(grd['actid'].unique()):
    ev_grd = grd[grd['actid'] == eid]
    pre = ev_grd[ev_grd['master'] == False]['pflood']
    flood = ev_grd[ev_grd['master'] == True]['pflood']
    if pre.empty or flood.empty:
        continue
    results.append({
        'actid': eid,
        'pre_mean': pre.mean(),
        'flood_mean': flood.mean(),
        'abs_diff': abs(pre.mean() - flood.mean()),
    })

pflood_comparison = pd.DataFrame(results)
n_identical = (pflood_comparison['abs_diff'] < 1e-10).sum()
n_total = len(pflood_comparison)

print("═══ pflood is a STATIC LABEL (not a temporal signal) ═══")
print(f"\nEvents with IDENTICAL mean pflood in pre-flood vs flood-time: "
      f"{n_identical} / {n_total} ({100*n_identical/n_total:.0f}%)")
print(f"\nThis means pflood encodes the ground-truth flood extent per tile,")
print(f"applied to ALL acquisitions (pre and post) as an ML label.")
print(f"It does NOT vary with acquisition date.")
print(f"\n→ Consequence: pflood cannot be used to assess flood signal strength")
print(f"  per acquisition. It is valid ONLY for computing spatial flood extent.")

# Visualise: scatter plot of pre vs flood pflood means
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pflood_comparison['pre_mean'], pflood_comparison['flood_mean'],
           s=40, color='steelblue', edgecolors='black', linewidths=0.5, alpha=0.8)
ax.plot([0, 50], [0, 50], 'k--', linewidth=1, alpha=0.5, label='y = x (perfect identity)')
ax.set_xlabel('Mean pflood — pre-flood acquisitions (%)')
ax.set_ylabel('Mean pflood — flood-time acquisitions (%)')
ax.set_title('pflood is identical in pre-flood and flood-time\n'
             '(confirms it is a static spatial label, not a temporal signal)')
ax.legend(fontsize=9)
ax.set_aspect('equal')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# TEMPORAL GAPS: pre-flood → flood-time acquisition spacing
# ═══════════════════════════════════════════════════════════════════════════════
# Since pflood is static, the key variable per event is the TEMPORAL STRUCTURE:
# how far apart are pre-flood and flood-time acquisitions?

acq = (
    gdf[gdf['crank'] == 1]
    .assign(source_date=lambda d: pd.to_datetime(d['source_date']))
    .groupby(['actid', 'master', 'source_date'], as_index=False)
    .agg(n_patches=('pflood', 'size'))
)

# Compute gap per event
gaps = []
for eid in sorted(acq['actid'].unique()):
    ev_acq = acq[acq['actid'] == eid]
    pre_dates = ev_acq[ev_acq['master'] == False]['source_date']
    flood_dates = ev_acq[ev_acq['master'] == True]['source_date']
    if pre_dates.empty or flood_dates.empty:
        continue
    gap_days = (flood_dates.min() - pre_dates.max()).days
    gaps.append({'actid': eid, 'gap_days': gap_days,
                 'pre_last': pre_dates.max(), 'flood_first': flood_dates.min()})

gap_df = pd.DataFrame(gaps).sort_values('gap_days', ascending=False).reset_index(drop=True)

print("═══ Temporal gaps: last pre-flood → first flood-time (GRD) ═══\n")
print(f"  Median gap: {gap_df['gap_days'].median():.0f} days")
print(f"  Mean gap:   {gap_df['gap_days'].mean():.0f} days")
print(f"  Max gap:    {gap_df['gap_days'].max()} days (Event {gap_df.iloc[0]['actid']})")
print(f"\n  Events with gap > 6 months: {(gap_df['gap_days'] > 180).sum()} / {len(gap_df)}")
print(f"  Events with gap > 1 year:   {(gap_df['gap_days'] > 365).sum()} / {len(gap_df)}")

# Histogram + strip chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [1.3, 1]})

# Left: histogram of gaps
ax1.hist(gap_df['gap_days'], bins=20, color='steelblue', edgecolor='black', alpha=0.8)
ax1.axvline(180, color='orangered', linestyle='--', linewidth=1.5, label='6-month threshold')
ax1.axvline(365, color='darkred', linestyle='--', linewidth=1.5, label='1-year threshold')
ax1.set_xlabel('Gap (days)')
ax1.set_ylabel('Number of events')
ax1.set_title('Distribution of pre→flood temporal gaps')
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

# Right: sorted bar chart highlighting problematic events
gap_sorted = gap_df.sort_values('gap_days').reset_index(drop=True)
colors = ['orangered' if g > 365 else 'darkorange' if g > 180 else 'steelblue'
          for g in gap_sorted['gap_days']]
ax2.barh(range(len(gap_sorted)), gap_sorted['gap_days'], color=colors,
         edgecolor='black', linewidth=0.3, alpha=0.8)
ax2.axvline(180, color='orangered', linestyle='--', linewidth=1, alpha=0.7)
ax2.axvline(365, color='darkred', linestyle='--', linewidth=1, alpha=0.7)
ax2.set_xlabel('Gap (days)')
ax2.set_ylabel('Events (sorted)')
ax2.set_title('Per-event temporal gap (sorted)')
ax2.set_yticks([])
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Show the worst offenders
print("\nTop 10 events with largest gaps:")
print(gap_df[['actid', 'pre_last', 'flood_first', 'gap_days']].head(10).to_string(index=False))


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# ALL EVENTS: Gantt-style timeline (temporal structure only)
# ═══════════════════════════════════════════════════════════════════════════════
# Since pflood is constant across acquisitions, dot size is meaningless for
# temporal signal. Instead, show a clean timeline with role markers + gap flags.

# Sort events by flood-time date (same as the schematic Gantt above)
event_order = (
    acq[acq['master'] == True]
    .groupby('actid')['source_date'].min()
    .sort_values()
    .index.tolist()
)
event_rank = {eid: i for i, eid in enumerate(event_order)}

# Merge gap info
gap_lookup = dict(zip(gap_df['actid'], gap_df['gap_days']))

fig, ax = plt.subplots(figsize=(16, 12))

for eid, grp in acq.groupby('actid'):
    if eid not in event_rank:
        continue
    y = event_rank[eid]
    grp_sorted = grp.sort_values('source_date').reset_index(drop=True)

    # connecting line
    ax.plot(grp_sorted['source_date'], [y] * len(grp_sorted),
            color='#cccccc', linewidth=0.8, zorder=1)

    # Separate pre-flood and flood-time
    pre_grp = grp_sorted[grp_sorted['master'] == False]
    flood_grp = grp_sorted[grp_sorted['master'] == True]

    # Pre-flood: open circles
    ax.scatter(pre_grp['source_date'], [y] * len(pre_grp),
               s=40, facecolors='white', edgecolors='steelblue', linewidths=1.2, zorder=2)
    # Flood-time: filled circles
    ax.scatter(flood_grp['source_date'], [y] * len(flood_grp),
               s=50, color='steelblue', edgecolors='black', linewidths=0.5, zorder=3)

    # Flag events with large gap
    gap = gap_lookup.get(eid, 0)
    if gap > 180:
        last_date = grp_sorted['source_date'].max()
        color = 'darkred' if gap > 365 else 'orangered'
        ax.text(last_date, y, f'  ⚠ {gap}d', ha='left', va='center',
                fontsize=7, color=color, style='italic', fontweight='bold', zorder=4)

ax.set_yticks(range(len(event_order)))
ax.set_yticklabels([f"Event {e}" for e in event_order], fontsize=7)
ax.set_xlabel('Date')
ax.set_title(
    'All 43 events: SAR acquisition timeline (GRD)\n'
    '○ = pre-flood  |  ● = flood-time  |  ⚠ = gap > 6 months',
    fontsize=10
)
ax.grid(axis='x', alpha=0.3)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
           markeredgecolor='steelblue', markersize=8, label='Pre-flood acquisition'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue',
           markeredgecolor='black', markersize=8, label='Flood-time acquisition'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()


## 6. Summary & Conclusions

### Delivered metadata fields

| Field | Status | Derivation |
|-------|--------|-----------|
| `flood_case` | ✓ Available | `KuroSiwo_<actid>` |
| `date_start` | ⚠ Proxy | Earliest pre-flood `source_date` (observational window start, not flood onset) |
| `date_end` | ⚠ Proxy | Flood-time `source_date` (SAR acquisition date, not flood end) |
| `bounding_box` | ✓ Available | Union of all tile geometries → WGS84 total_bounds |
| `max_flood_extent_km²` | ⚠ Approximation | `sum(pflood × tile_area)` over flood-time GRD tiles, EPSG:6933 |
| `date_of_max_flood_extent` | ✗ Not derivable | Set to `date_end` (only 1 flood-time acquisition per event) |

### Critical finding: `pflood` is a static spatial label

`pflood` is **identical** for pre-flood and flood-time patches (all 43 events). It encodes the ground-truth flood extent per tile for ML training — it does NOT vary with acquisition date.

### Key limitations

1. **No flood timeline.** Typically 2 pre-flood + 1 flood-time acquisition per event.
2. **Flood onset/end unknown.** Dates are SAR acquisition dates, not hydrological boundaries.
3. **Large temporal gaps.** 10/43 events have >6-month gaps; 2 exceed 1 year (max 672 days).
4. **Variable tile sizes.** Tiles range from 1.3–4.9 km² (CV=33%); geometry-derived areas required.
5. **Sparse labeling.** Only ~11% of flood-time patches have non-NaN pflood; of those, 62% are zero.
